# 10 — Baseline v2 no pH (EF01): protocolo L=2304 + purge/embargo + 5 fatias

Baselines clássicos **idênticos ao 00** (persistência · sazonal-naive-288 · média-móvel-288 · ARIMA(2,1,2) em grade horária · Prophet opcional), agora no **protocolo v2**: `L=2304 → H=288` (8 d → 1 d, passo 5 min), interpolação limite 24, descarte de janelas com NaN, val em **5 fatias de 2024 por data de fim** (19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão — nova]**) com **purge/embargo ±H** no treino. 2025 intocado (benchmark futuro). Modelos determinísticos, sem seeds.

## Protocolo v2 × v1

| | v1 (00) | v2 (este notebook) |
|---|---|---|
| `L → H` | 8640 → 288 (30 d) | **2304 → 288 (8 d)** |
| Val | 4 fatias (abr/jul/set/nov) | **5 fatias: as 4 + 13–22/dez (verão, p/ o NNLS futuro; aqui só reporte)** |
| Split | por data de fim, sem purge (gap mín −288 passos: alvo de treino dentro da val) | **por data de fim + purge: treino exclui janelas cujo alvo `[fim−H, fim]` intersecte qualquer fatia estendida ±H (gap mín +289 passos)** |
| Baselines | persist/sazonal-288/MM-288/ARIMA-h/Prophet | **idênticos** |
| Esperado pH (de `/tmp/v2split/validate_split.py`) | treino 24.659 + val 10.080 | **treino ~59.349 + val 12.960 ([2880, 2880, 2880, 1440, 2880]) + purge ~3.311 + NaN-desc ~26.910** |

Lógica de split/purge reaproveitada **verbatim** de `/tmp/v2split/validate_split.py` (`val_mask`/`purge_train`/`signed_gap_steps`) — ver célula §5.

## Execução remota (UM job por vez — 12c/23 GB estouram com concorrência)

- **Solo (máquina livre):** `.venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=2400 notebooks/10-v2-baseline-ph.ipynb`
- **Compartilhada:** `OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=2400 notebooks/10-v2-baseline-ph.ipynb`
- Nunca `sleep` dentro do comando remoto (o canal MCP expira); `pkill -f` com o truque `[.]`.
- Tempo estimado: **10–25 min** (ARIMA reestima por origem; Prophet ~minutos; o resto é vetorizado).
- Prophet pula sem CmdStan (comportamento preservado do 00); ARIMA com fallback honesto p/ persistência.

## Saídas (criadas pela execução)

`resultados/10-v2-baseline-ph/`: `metricas_treino.csv` · `metricas_val.csv` (primária) · `metricas_val_diaria.csv` (45 dias-âncora) · `metricas_por_dia.csv` (45 linhas, incl. dez) · **`metricas_por_fatia.csv` (novo: 5 linhas × modelos, incl. dez)** · `modelos/arima212_cauda_treino.pkl` + `modelos/prophet_ph.json` (só o que o 00 salva; gitignored, vão ao Release) · `figs/` 01-eda/02-limpeza/03-stl/04-forecasts/05-mae/06-val-dias (equivalentes às do 00). O `README.md` do experimento (mesmo formato do 00 + seção “Protocolo v2”) é escrito **após** a execução, com números reais + procedência remota (host + work dir).

Convenção: nada in-place em 00–08 · nada de `src/` · nada de 2025 neste notebook.

In [1]:
import os
import pickle
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_ph_2024.csv"
OUT = ROOT / "resultados" / "10-v2-baseline-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# Protocolo v2 (travado). Baselines determinísticos — sem seeds.
L, H = 2304, 288  # 8 d -> 1 d (passo 5 min)
SEASON = 288
INTERP_LIMIT = 24  # 2 h
# val: 4 fatias do 00 + 13-22/dez (10 d, verão austral — p/ o NNLS futuro; aqui só reporte)
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24"),
              ("2024-12-13", "2024-12-22")]
ARIMA_ORDER = (2, 1, 2)
ARIMA_STRIDE = 48  # idêntico ao 00 (custo)
print("ROOT:", ROOT, "| CSV existe:", CSV.exists())
print(f"L={L} H={H} fatias={len(VAL_SLICES)} ARIMA={ARIMA_ORDER} stride={ARIMA_STRIDE}")
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
      "| cpu:", os.cpu_count())
print("OUT:", OUT)

ROOT: /home/marcos/temporal-model | CSV existe: True
L=2304 H=288 fatias=5 ARIMA=(2, 1, 2) stride=48
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>'} | cpu: 12
OUT: /home/marcos/temporal-model/resultados/10-v2-baseline-ph


## 1. Carga

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 11665 (11.1%)


,ds,y
count,105121,93456.00000
mean,2024-07-01 12:00:00,5.90729
min,2024-01-01 00:00:00,5.21000
25%,2024-04-01 06:00:00,5.71000
50%,2024-07-01 12:00:00,5.92000
75%,2024-09-30 18:00:00,6.09000
max,2024-12-31 00:00:00,6.68000
std,NaN,0.30356


## 2. EDA — perfil, faltantes e ciclo diário

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("ph EF01 2024 — série completa (treino)")
ax[0].set_ylabel("ph")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")

maior gap: 4971 passos = 414.2 h | gaps > 24 passos: 9


fig salva


## 3. Limpeza — grade completa + interpolação limitada

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
print(f"blocos NaN pós-interp: {len(blocos)}")
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots = {len(g)*5/60:.1f} h)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 6408
blocos NaN pós-interp: 9
  outage 2024-01-16 11:35:00 → 2024-01-18 17:50:00 (652 slots = 54.3 h)
  outage 2024-02-13 17:00:00 → 2024-02-13 17:05:00 (2 slots = 0.2 h)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots = 1.1 h)
  outage 2024-04-29 11:30:00 → 2024-05-02 03:10:00 (765 slots = 63.8 h)
  outage 2024-05-27 12:00:00 → 2024-06-13 16:10:00 (4947 slots = 412.2 h)
  outage 2024-10-19 06:05:00 → 2024-10-19 06:30:00 (6 slots = 0.5 h)
  outage 2024-10-19 13:35:00 → 2024-10-19 13:50:00 (4 slots = 0.3 h)
  outage 2024-11-25 15:15:00 → 2024-11-25 15:20:00 (2 slots = 0.2 h)
  outage 2024-12-03 13:30:00 → 2024-12-03 14:50:00 (17 slots = 1.4 h)


fig salva


## 4. Estacionariedade (ADF) e decomposição STL (trecho limpo jul–ago)

In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-1.79 p-valor=0.384 → NÃO estacionária


fig salva


## 5. Janelamento (L=2304) + val em 5 fatias + purge/embargo ±H

Janelas por data de **fim** (sem shuffle); descarte com NaN pós-interp (idêntico ao 00). Treino = janelas válidas fora da val que **sobrevivem ao purge**: exclui quem tem o alvo `[fim−H, fim]` intersectando qualquer fatia estendida `±H` (replica `purge_train` de `/tmp/v2split/validate_split.py`). Trava se o purge falhar (gap < H+1 ou overlap > 0).

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
n_desc_nan = int((~ok).sum())
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date

# --- val por data de fim (5 fatias, incl. dez) ---
is_val = np.zeros(len(ends), dtype=bool)
per_slice = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    per_slice.append(int(m.sum()))
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")


def purge_train(ends, is_val, slices):
    """Descarta treino cujo alvo [fim-H, fim] intersecte fatia estendida ±H.
    Verbatim de /tmp/v2split/validate_split.py (H global)."""
    keep = is_val.copy()
    drop = np.zeros(len(ends), dtype=bool)
    for a, b in slices:
        A = pd.Timestamp(a) - pd.Timedelta(minutes=5 * H)          # ini-H
        B = pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) \
            + pd.Timedelta(minutes=5 * H)                           # fim+H
        tgt0 = ends - pd.Timedelta(minutes=5 * H)
        hit = (~is_val) & (ends >= A) & (tgt0 <= B)  # alvo ∩ [A,B] ≠ ∅
        drop |= hit
    keep[~is_val & ~drop] = True  # treino sobrevivente
    return keep, drop  # keep=True → val ou treino válido


def signed_gap_steps(ends_tr, slices):
    """Distância (passos 5min) do alvo [fim-H,fim] à fatia mais próxima; <0 = overlap.
    Verbatim de /tmp/v2split/validate_split.py."""
    if not len(ends_tr):
        return None
    e = ends_tr.values.astype("datetime64[m]").astype(np.int64)  # min
    best = np.full(len(e), 10 ** 12)
    for a, b in slices:
        A = (pd.Timestamp(a).to_datetime64().astype("datetime64[m]").astype(int))
        B = ((pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
             .to_datetime64().astype("datetime64[m]").astype(int))
        s0 = e - H * 5
        ov = (e >= A) & (s0 <= B)
        gap = np.where(e < A, (A - e) // 5, np.where(s0 > B, (s0 - B) // 5, -(np.minimum(e, B) - np.maximum(s0, A)) // 5 - 1))
        best = np.minimum(best, gap)
    return best


keep, drop = purge_train(ends, is_val, VAL_SLICES)
va = np.where(is_val)[0]
tr = np.where(keep & ~is_val)[0]
print(f"treino (pós-purge): {len(tr)} | val: {len(va)} | "
      f"descartadas (NaN): {n_desc_nan} | purge: {int(drop.sum())}")

# --- asserts de cobertura por fatia (pH: fatias limpas → val cheia) ---
esperado = [288 * ((pd.Timestamp(b) - pd.Timestamp(a)).days + 1) for a, b in VAL_SLICES]
assert per_slice == esperado, f"cobertura por fatia fora do esperado: {per_slice} vs {esperado}"
assert len(va) == sum(esperado) == 12960, f"val total inesperada: {len(va)}"
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
assert int(drop.sum()) > 0, "purge removeu zero janelas — lógica inativa?"

# --- trava do purge: gap mín ≥ H+1 e overlap zero ---
gaps = signed_gap_steps(ends[tr], VAL_SLICES)
print(f"gap mín alvo-treino→val: +{int(gaps.min())} passos (exigido ≥ {H + 1}); "
      f"treino c/ alvo∩val: {int((gaps < 0).sum())}")
assert int((gaps < 0).sum()) == 0, "LEAKAGE: há alvo de treino dentro da val!"
assert int(gaps.min()) >= H + 1, f"purge/embargo falhou: gap {int(gaps.min())} < {H + 1}"

# --- dias-âncora (23:55) na val: 45 = 10+10+10+5+10 ---
daily_mask = (ends.time == pd.Timestamp("23:55").time()) & is_val
daily_idx = np.where(daily_mask)[0]
por_dia_ct = [int(((ends[daily_idx].date >= pd.Timestamp(a).date()) &
                   (ends[daily_idx].date <= pd.Timestamp(b).date())).sum())
              for a, b in VAL_SLICES]
print("dias-âncora na val:", len(daily_idx), "por fatia:", por_dia_ct)
assert len(daily_idx) == 45, f"dias-âncora inesperados: {len(daily_idx)}"
assert por_dia_ct == [10, 10, 10, 5, 10], f"âncoras por fatia: {por_dia_ct}"

# série de ajuste: fatias de val removidas (Prophet/ARIMA)
s_train = s.copy()
for a, b in VAL_SLICES:
    s_train.loc[a:b] = np.nan
print("slots de treino p/ ajuste:", int(s_train.notna().sum()))

fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
fatia 2024-12-13 → 2024-12-22: 2880 janelas válidas
treino (pós-purge): 59349 | val: 12960 | descartadas (NaN): 26910 | purge: 3311
gap mín alvo-treino→val: +289 passos (exigido ≥ 289); treino c/ alvo∩val: 0
dias-âncora na val: 45 por fatia: [10, 10, 10, 5, 10]
slots de treino p/ ajuste: 85753


## 6. Baselines baratos (treino rolante + val)

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
pred_tr = cheap_preds(Xtr)
pred_va = cheap_preds(Xva)
print("treino rolante:")
print(pd.DataFrame({m: metricas(Ytr, p) for m, p in pred_tr.items()}).T.round(4).to_string())
print("val rolante:")
print(pd.DataFrame({m: metricas(Yva, p) for m, p in pred_va.items()}).T.round(4).to_string())

treino rolante:


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0580  0.0796  0.9967  0.9960
sazonal_naive_288  0.0551  0.0757  0.9491  0.9479
media_movel_288    0.0533  0.0715  0.9145  0.9135
val rolante:


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0599  0.0829  1.0229  1.0221
sazonal_naive_288  0.0406  0.0589  0.6990  0.6982
media_movel_288    0.0497  0.0652  0.8478  0.8476


## 7. ARIMA em grade horária (val com stride — custo)

In [8]:
hs = s.resample("1h").mean()

def arima_hora(e):
    he = e.floor("h")
    ctx = hs.loc[he - pd.Timedelta(hours=719):he].values
    fc = ARIMA(ctx, order=ARIMA_ORDER).fit().get_forecast(24).predicted_mean.values
    return np.repeat(fc, 12)[:H]

def roda_arima(idxs, nome):
    P = np.empty((len(idxs), H))
    t0 = time.time()
    for j, i in enumerate(idxs):
        try:
            P[j] = arima_hora(ends[i])
        except Exception:
            P[j] = np.repeat(X[i, -1], H)
        if (j + 1) % 10 == 0:
            print(f"  {nome}: {j+1}/{len(idxs)} origens...", flush=True)
    print(f"ARIMA {nome}: {len(idxs)} origens em {time.time()-t0:.0f}s")
    return P

idx_a = va[::ARIMA_STRIDE]
Pa = roda_arima(idx_a, "val")
print("ARIMA val:", metricas(Y[idx_a], Pa))
Pd = roda_arima(daily_idx, "val-diaria")
print("ARIMA dias-âncora:", metricas(Y[daily_idx], Pd))

h_tr = s_train.resample("1h").mean().dropna().iloc[-720:].values
with open(OUT / "modelos" / "arima212_cauda_treino.pkl", "wb") as f:
    pickle.dump(ARIMA(h_tr, order=ARIMA_ORDER).fit(), f)
print("modelo salvo")

  val: 10/270 origens...


  val: 20/270 origens...


  val: 30/270 origens...


  val: 40/270 origens...


  val: 50/270 origens...


...

  val-diaria: 10/45 origens...


  val-diaria: 20/45 origens...


  val-diaria: 30/45 origens...


  val-diaria: 40/45 origens...


ARIMA val-diaria: 45 origens em 12s
ARIMA dias-âncora: {'MAE': 0.06923630952380948, 'RMSE': 0.09291483317761566, 'MAPE': 1.1911480191038282, 'sMAPE': 1.1804077468987606}
modelo salvo


## 8. Prophet (opcional — pula se `prophet`/CmdStan indisponível; ajuste só no treino)

In [9]:
PROPHET_OK = False
try:
    from prophet import Prophet
    import cmdstanpy
    assert cmdstanpy.cmdstan_path() is not None
    df_train = pd.DataFrame({"ds": s_train.index, "y": s_train.values}).dropna()
    m = Prophet(daily_seasonality=True, weekly_seasonality=True)
    m.fit(df_train)
    fmap = m.predict(pd.DataFrame({"ds": s.index})).set_index("ds")["yhat"]

    def fatia(idxs):
        E = ends[idxs]
        return np.stack([[fmap.loc[d - pd.Timedelta(minutes=5*(H-1-h))] for h in range(H)] for d in E])

    Pp_va, Pp_d = fatia(va), fatia(daily_idx)
    PROPHET_OK = True
    print("Prophet val:", metricas(Yva, Pp_va))
    from prophet.serialize import model_to_json
    (OUT / "modelos" / "prophet_ph.json").write_text(model_to_json(m))
    print("modelo salvo")
except Exception as e:
    print(f"Prophet pulado ({type(e).__name__}: {str(e)[:150]}).")

Importing plotly failed. Interactive plots will not work.


19:36:42 - cmdstanpy - INFO - Chain [1] start processing


19:37:34 - cmdstanpy - INFO - Chain [1] done processing


Prophet val: {'MAE': 0.14627676460818187, 'RMSE': 0.18850845293326868, 'MAPE': 2.5192521567826147, 'sMAPE': 2.498663228075736}
modelo salvo


## 9. Comparação final + val dia a dia + por fatia (incl. dez)

In [10]:
linhas = {m: metricas(Yva, p) for m, p in pred_va.items()}
linhas["arima_212_h"] = metricas(Y[idx_a], Pa)
if PROPHET_OK:
    linhas["prophet"] = metricas(Yva, Pp_va)
tab_va = pd.DataFrame(linhas).T.round(4)
tab_va.to_csv(OUT / "metricas_val.csv")
tab_tr = pd.DataFrame({m: metricas(Ytr, p) for m, p in pred_tr.items()}).T.round(4)
tab_tr.to_csv(OUT / "metricas_treino.csv")
print("=== treino rolante ===")
print(tab_tr.to_string())
print("=== val rolante (5 fatias, protocolo v2) ===")
print(tab_va.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in pred_tr}
diario["arima_212_h"] = metricas(Yd, Pd)
if PROPHET_OK:
    diario["prophet"] = metricas(Yd, Pp_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_val_diaria.csv")
print("=== val dias-âncora (45) ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in pred_tr},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["arima_212_h"] = [mae(Yd[k:k+1], Pd[k:k+1]) for k in range(len(Yd))]
if PROPHET_OK:
    por_dia["prophet"] = [mae(Yd[k:k+1], Pp_d[k:k+1]) for k in range(len(Yd))]
por_dia.to_csv(OUT / "metricas_por_dia.csv")
assert len(por_dia) == 45 and (por_dia.index >= "2024-12-13").sum() == 10, \
    f"por_dia sem cobertura de dez: {len(por_dia)} linhas"
print(por_dia.round(4).to_string())

# --- por fatia (NOVO no v2): uma linha por (fatia, modelo), incl. dez ---
va_ends = ends[va]
rows = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m_va = (va_ends.date >= d0) & (va_ends.date <= d1)
    ii = va[m_va]  # índices globais da fatia
    for mod, P in pred_va.items():
        mm = metricas(Y[ii], P[m_va])
        rows.append({"fatia": f"{a}→{b}", "modelo": mod, **mm})
    m_a = (ends[idx_a].date >= d0) & (ends[idx_a].date <= d1)
    rows.append({"fatia": f"{a}→{b}", "modelo": "arima_212_h",
                 **metricas(Y[idx_a[m_a]], Pa[m_a])})
    if PROPHET_OK:
        rows.append({"fatia": f"{a}→{b}", "modelo": "prophet",
                     **metricas(Y[ii], Pp_va[m_va])})
tab_f = pd.DataFrame(rows, columns=["fatia", "modelo", "MAE", "RMSE", "MAPE", "sMAPE"]).round(4)
tab_f.to_csv(OUT / "metricas_por_fatia.csv", index=False)
assert tab_f["fatia"].nunique() == 5, "por_fatia sem as 5 fatias!"
assert (tab_f["fatia"] == "2024-12-13→2024-12-22").any(), "fatia dez ausente no por_fatia!"
print("=== val por fatia (MAE) ===")
print(tab_f.pivot(index="fatia", columns="modelo", values="MAE").to_string())
print(f"\nRégua treino: {tab_tr['MAE'].idxmin()} = {tab_tr['MAE'].min():.4f}")
print(f"Régua val: {tab_va['MAE'].idxmin()} = {tab_va['MAE'].min():.4f}")

=== treino rolante ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0580  0.0796  0.9967  0.9960
sazonal_naive_288  0.0551  0.0757  0.9491  0.9479
media_movel_288    0.0533  0.0715  0.9145  0.9135
=== val rolante (5 fatias, protocolo v2) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0599  0.0829  1.0229  1.0221
sazonal_naive_288  0.0406  0.0589  0.6990  0.6982
media_movel_288    0.0497  0.0652  0.8478  0.8476
arima_212_h        0.0602  0.0842  1.0291  1.0281
prophet            0.1463  0.1885  2.5193  2.4987
=== val dias-âncora (45) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0692  0.0929  1.1911  1.1804
sazonal_naive_288  0.0412  0.0594  0.7089  0.7080
media_movel_288    0.0487  0.0637  0.8316  0.8314
arima_212_h        0.0692  0.0929  1.1911  1.1804
prophet            0.1525  0.1966  2.6307  2.6064
            persistencia  sazonal_naive_288  media_movel_288  arima_212_h  prophet
2024-04-19        0.

=== val por fatia (MAE) ===
modelo                 arima_212_h  media_movel_288  persistencia  prophet  sazonal_naive_288
fatia                                                                                        
2024-04-19→2024-04-28       0.0273           0.0267        0.0287   0.0937             0.0282
2024-07-20→2024-07-29       0.0419           0.0392        0.0434   0.1032             0.0391
2024-09-15→2024-09-24       0.1016           0.0768        0.0983   0.2912             0.0693
2024-11-20→2024-11-24       0.0542           0.0421        0.0536   0.2467             0.0215
2024-12-13→2024-12-22       0.0729           0.0598        0.0721   0.0468             0.0354

Régua treino: media_movel_288 = 0.0533
Régua val: sazonal_naive_288 = 0.0406


## 10. Figuras

In [11]:
E = ends[tr]
ks = [0, len(Xtr) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, ks):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xtr[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Ytr[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_tr["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred_tr["persistencia"][k], ":", lw=1, label="persistência")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab_va["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE na val (5 fatias, v2) — baselines (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, ax = plt.subplots(figsize=(12, 3.5))
pdf = por_dia
for col, ls in [("sazonal_naive_288", "--"), ("persistencia", ":"), ("media_movel_288", "-."), ("arima_212_h", "-")]:
    if col in pdf.columns:
        ax.plot(pd.to_datetime(pdf.index), pdf[col], ls, lw=1.1, label=col)
ax.set_title("ph — MAE por dia-âncora na val (5 fatias sazonais, v2, incl. dez)")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")
print("figs salvas")

figs salvas


## 11. Conclusões (preencher com números reais após a execução)

Réguas do protocolo v2 acima (`metricas_val.csv` = primária; `metricas_por_fatia.csv` mostra cada fatia, incl. dez). O sazonal lag-365 segue fora (incalculável sem 2023). Próximo: portar o mesmo protocolo v2 p/ o OD.

### Protocolo v2 (resumo p/ o README do experimento)

- Janelas `L=2304 → H=288` (8 d → 1 d, 5 min), interp `time` limite 24, descarte com NaN.
- Val = 5 fatias por data de fim (19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]**); a 5ª fatia cobre o verão austral (estação sem cobertura na VAL4) e servirá ao NNLS futuro — aqui é só reporte.
- Purge/embargo: treino exclui janelas cujo alvo `[fim−H, fim]` intersecte qualquer fatia estendida `±H` (gap mín +289 passos; v1 era −288). Ver célula §5 (trava por `assert`).
- Baselines idênticos ao 00, determinísticos, sem seeds; ARIMA em grade horária 720 h/24 h repeat ×12; Prophet pula sem CmdStan.

### Procedência da execução (preencher no commit da execução)

- Host remoto: `<host>` · work dir: `<dir>` · data: `<aaaa-mm-dd>` · threads: `<solo|compartilhada + env>`
- Pós-execução: escrever `resultados/10-v2-baseline-ph/README.md` (formato do 00 + seção “Protocolo v2”), indexar em `resultados/README.md` + `notebooks/README.md` + README §7 — com números reais.